A notebook to fine-tune a GLiNER model based on [this example](https://github.com/urchade/GLiNER/blob/main/examples/finetune.ipynb).

## Env Install

```shell
conda create -n gliner_finetune python=3.10     
conda activate gliner_finetune
conda install gliner accelerate seqeval datasets
```

## Data and Libraries

In [1]:
import json
import random

from datasets import load_dataset

from seqeval.metrics.sequence_labeling import get_entities

import re

from collections import Counter
# import matplotlib.pyplot as plt
import pandas as pd
from typing import List, Dict

from dataset_processing import *

c:\Users\Desktop\.conda\envs\gliner_finetune\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading is based on [er_combine_data](CWED4ETA/GLINER/er_combine_data.ipynb) notebook.

### Specific Dataset Loading

#### Climate-Change-NER (IBM)

In [3]:
ibmccner = load_ibmccner(["train", "validation"])

Loading 'ibm-research/Climate-Change-NER' with splits: ['train', 'validation']


KeyboardInterrupt: 

#### BiodivNER

In [2]:
biodivner = load_biodivner(["train", "validation"])

Loading '/home/p0l3/RAD/CWED4ETA/CWED4ETA/CWED4ETA/DATA/BiodivNER' with splits: ['train', 'validation']
Converting from BIO tags to char spans.
Processing file: /home/p0l3/RAD/CWED4ETA/CWED4ETA/CWED4ETA/DATA/BiodivNER/dev.csv...
Processing file: /home/p0l3/RAD/CWED4ETA/CWED4ETA/CWED4ETA/DATA/BiodivNER/train.csv...
Converting from char spans to token spans.


#### CWED4ETA

In [2]:
cwed4eta = load_cwed4eta()

Loading 'C:\Users\ANDRIJA_RAD\CLIRENER\CliReNER\DATA\LABEL_STUDIO\project-28-at-2025-11-11-12-33-f873727e.json' with no splits.
COnverting to desirable char spans.
COnverting from char spans to token spans.


### Data Preprocessing

In [3]:
# train_path = "data.json"

# with open(train_path, "r") as f:
#     data = json.load(f)

# Here choose the loaded dataset
data = cwed4eta

print('Dataset size:', len(data))

random.shuffle(data)
print('Dataset is shuffled...')

train_dataset = data[:int(len(data)*0.9)]
test_dataset = data[int(len(data)*0.9):]

print('Dataset is splitted...')

Dataset size: 1042
Dataset is shuffled...
Dataset is splitted...


In [4]:
print(data[1])

{'tokenized_text': ['However', ',', 'it', 'is', 'quite', 'challenging', 'to', 'implement', 'a', 'robust', 'real-time', 'fault', 'diagnosis', 'and', 'protection', 'scheme', 'to', 'ensure', 'battery', 'safety', 'and', 'performance', '.'], 'ner': [[9, 9, 'Other'], [10, 12, 'Method'], [14, 15, 'Method'], [18, 18, 'Physical Artefact'], [19, 19, 'Other'], [21, 21, 'Other']]}


In [5]:
i = random.randint(0, len(data)-1)
print(" ".join(train_dataset[i]["tokenized_text"]))
print(train_dataset[i]["tokenized_text"])
print(train_dataset[i]["ner"])
# print(train_dataset[i]["negative"])

for entity in  train_dataset[i]["ner"]:
    temp = train_dataset[i]["tokenized_text"][entity[0]:entity[1]+1] # The ranges are inclusive from both sides (thus entity[1] + 1).
    temp_context = train_dataset[i]["tokenized_text"][max(0, entity[0]-2):min(entity[1]+2, len(train_dataset[i]["tokenized_text"])-1)]
    print(f"Entry: {entity} ---  pilener[i][\"tokenized_text\"][{entity[0]}:{entity[1]}]: {temp} --- Context: {temp_context}")
    # 
    # 

Simulation results showed that the aquifer is chiefly influenced by the interaction with the Adige River and that the influence of anthropogenic activities on vulnerability of groundwater resources varies within the study area .
['Simulation', 'results', 'showed', 'that', 'the', 'aquifer', 'is', 'chiefly', 'influenced', 'by', 'the', 'interaction', 'with', 'the', 'Adige', 'River', 'and', 'that', 'the', 'influence', 'of', 'anthropogenic', 'activities', 'on', 'vulnerability', 'of', 'groundwater', 'resources', 'varies', 'within', 'the', 'study', 'area', '.']
[[0, 1, 'Intellectual Artefact'], [5, 5, 'Geographical Feature'], [11, 11, 'Other'], [14, 15, 'Body of Water'], [21, 22, 'Other'], [24, 24, 'Quantity'], [26, 27, 'Body of Water'], [31, 32, 'Location']]
Entry: [0, 1, 'Intellectual Artefact'] ---  pilener[i]["tokenized_text"][0:1]: ['Simulation', 'results'] --- Context: ['Simulation', 'results', 'showed']
Entry: [5, 5, 'Geographical Feature'] ---  pilener[i]["tokenized_text"][5:5]: ['aqu

In [6]:
print(" ".join(data[i]["tokenized_text"]))

Simulation results showed that the aquifer is chiefly influenced by the interaction with the Adige River and that the influence of anthropogenic activities on vulnerability of groundwater resources varies within the study area .


## Model Fine-Tuning

In [7]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "true"
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"]="python"

import torch
from gliner import GLiNERConfig, GLiNER
from gliner.training import Trainer, TrainingArguments
from gliner.data_processing.collator import DataCollatorWithPadding, DataCollator
from gliner.utils import load_config_as_namespace
from gliner.data_processing import WordsSplitter, GLiNERDataset

In [8]:
device = torch.device('cuda:0') if torch.cuda.is_available() else torch.device('cpu')

model = GLiNER.from_pretrained("urchade/gliner_medium-v2.1")

Fetching 5 files: 100%|██████████| 5/5 [00:00<?, ?it/s]
c:\Users\Desktop\.conda\envs\gliner_finetune\lib\site-packages\transformers\convert_slow_tokenizer.py:561: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


In [9]:
# use it for better performance, it mimics original implementation but it's less memory efficient
data_collator = DataCollator(model.config, data_processor=model.data_processor, prepare_labels=True)

In [10]:
# Optional: compile model for faster training
model.to(device)
print("done")

done


In [11]:
# calculate number of epochs
num_steps = 4000
batch_size = 8
data_size = len(train_dataset)
num_batches = data_size // batch_size
num_epochs = max(1, num_steps // num_batches)

training_args = TrainingArguments(
    output_dir="models/GLiNER_med_v2_1/CLIRENER_V1",
    learning_rate=5e-6,
    weight_decay=0.01,
    others_lr=1e-5,
    others_weight_decay=0.01,
    lr_scheduler_type="linear", #cosine
    warmup_ratio=0.1,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    focal_loss_alpha=0.75,
    focal_loss_gamma=2,
    num_train_epochs=num_epochs,
    eval_strategy="steps", # PC1, # evaluation_strategy="steps",
    save_steps = 100,
    save_total_limit=10,
    dataloader_num_workers = 0,
    use_cpu = False,
    report_to="none",
    )

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=model.data_processor.transformer_tokenizer,
    data_collator=data_collator,
)

trainer.train()

C:\Users\Desktop\AppData\Local\Temp\ipykernel_17108\1662017028.py:29: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Step,Training Loss,Validation Loss
500,25.512900,230.838409
1000,11.537300,155.501480
1500,8.650400,120.929977
2000,6.906700,111.872673
2500,5.916000,95.261406
3000,5.098300,93.605103
3500,4.809900,88.187828
4000,4.429700,87.738052


Skipping iteration due to error: cannot reshape tensor of 0 elements into shape [1, -1, 0] because the unspecified dimension size -1 can be any value and is ambiguous


TrainOutput(global_step=4012, training_loss=9.09250338032381, metrics={'train_runtime': 7807.8761, 'train_samples_per_second': 4.08, 'train_steps_per_second': 0.514, 'total_flos': 0.0, 'train_loss': 9.09250338032381, 'epoch': 34.0})

## Inference

In [13]:
trained_model = GLiNER.from_pretrained("models/GLiNER_med_v2_1/CLIRENER_V1/checkpoint-4012", load_tokenizer=True)

config.json not found in C:\Users\ANDRIJA_RAD\CLIRENER\CliReNER\FINETUNES\GLINER\models\GLiNER_med_v2_1\CLIRENER_V1\checkpoint-4012


In [20]:
text = """Resulting regional commitments would be around 10% higher than the global signal for the vulnerable Pacific region, mainly due to higher relative Antarctic contributions."""
LABELS = CLIRENER_LABELS_V1
# Labels for entity prediction
labels = list(LABELS)
# ["Person", "Award"] # for v2.1 use capital case for better performance

# Perform entity prediction
entities = trained_model.predict_entities(text, labels, threshold=0.1)

# Display predicted entities and their labels
print(text)
for entity in entities:
    print(entity["text"], "=>", entity["label"])

Resulting regional commitments would be around 10% higher than the global signal for the vulnerable Pacific region, mainly due to higher relative Antarctic contributions.
Resulting => Other
regional commitments => Policy
10% => Quantity
higher => Quantity
global signal => Physical Phenomenon
vulnerable => Other
Pacific region => Location
higher => Quantity
relative => Quantity
Antarctic contributions => Other
